In [1]:
import pickle

import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import center_in_box, unwrap, wrap
from rdkit import Chem

symbol_to_atomic_number = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "S": 16,
}

In [2]:
mols = [
    "1_4-dioxane",
    "1-propanethiol",
    "2-pentanone",
    "aceticacid",
    "acetonitrile",
    "aniline",
    "benzamide",
    "benzene",
    "ethylphenylether",
    "methanol",
    "methylbenzoate",
    "methylpentanoate",
    "m-hydroxybenzaldehyde",
    "naphthalene",
    "n-octane",
    "n-pentane",
    "o-cresol",
    "pyridine",
    "thioanisole",
    "water",
]

smiles = [
    "C1COCCO1",
    "CCCS",
    "CCCC(C)=O",
    "CC(O)=O",
    "CC#N",
    "Nc1ccccc1",
    "NC(=O)c1ccccc1",
    "c1ccccc1",
    "CCOc1ccccc1",
    "CO",
    "COC(=O)c1ccccc1",
    "CCCCC(=O)OC",
    "Oc1cccc(C=O)c1",
    "c1ccc2ccccc2c1",
    "CCCCCCCC",
    "CCCCC",
    "Cc1ccccc1O",
    "c1ccncc1",
    "CSc1ccccc1",
    "O",
]

In [3]:
z = []
xyz_mm = []
xyz_qm = []
charges_mm = []

base_path = (
    "/scratch1/joao/tip3p_final/simulations/mm_sol/rep1/reweighting/reweighting_data/"
)
for name, smiles in zip(mols, smiles):
    pdb = base_path + f"{name}.pdb"
    dcd = base_path + f"{name}.dcd"

    # Get number of atoms of solute
    mol = Chem.MolFromSmiles(smiles)
    mol_H = Chem.AddHs(mol)
    solute_n_atoms = mol_H.GetNumAtoms()

    print(f"Processing {name} with {solute_n_atoms} atoms.")

    # Create the Universe
    u = mda.Universe(pdb, dcd)
    solute = u.select_atoms(f"index 0:{solute_n_atoms - 1}")
    solvent = u.select_atoms(f"not index 0:{solute_n_atoms - 1}")

    # Define the transformations
    transforms = [
        unwrap(solute + solvent),
        center_in_box(solute, wrap=False),
        wrap(solvent + solute),
    ]

    # Apply the transformations
    u.trajectory.add_transformations(*transforms)

    for ts in u.trajectory[::120]:
        z.append(np.asarray([symbol_to_atomic_number[atom.element] for atom in solute]))
        xyz_qm.append(np.asarray(solute.positions))
        xyz_mm.append(np.asarray(solvent.positions))
        charges_mm.append(
            np.asarray([-0.834 if atom.element == "O" else 0.417 for atom in solvent])
        )

Processing 1_4-dioxane with 14 atoms.


/home/joaomorado/micromamba/envs/fes-ml-aev/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Processing 1-propanethiol with 12 atoms.
Processing 2-pentanone with 16 atoms.
Processing aceticacid with 8 atoms.
Processing acetonitrile with 6 atoms.
Processing aniline with 14 atoms.
Processing benzamide with 16 atoms.
Processing benzene with 12 atoms.
Processing ethylphenylether with 19 atoms.
Processing methanol with 6 atoms.
Processing methylbenzoate with 18 atoms.
Processing methylpentanoate with 20 atoms.
Processing m-hydroxybenzaldehyde with 15 atoms.
Processing naphthalene with 18 atoms.
Processing n-octane with 26 atoms.
Processing n-pentane with 17 atoms.
Processing o-cresol with 16 atoms.
Processing pyridine with 11 atoms.
Processing thioanisole with 16 atoms.
Processing water with 3 atoms.


In [4]:
# Save the processed data
data_dict = {
    "z": z,
    "xyz_qm": xyz_qm,
    "xyz_mm": xyz_mm,
    "charges_mm": charges_mm,
}

with open("processed_traj_data.pkl", "wb") as f:
    pickle.dump(data_dict, f)